In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.chains import RetrievalQA
from langchain.llms import HuggingFacePipeline
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

def create_advanced_rag(documents: list, model_name: str = "microsoft/DialoGPT-medium"):
    """
    Create an advanced RAG system using LangChain
    """
    # Initialize embeddings
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        model_kwargs={'device': 'cpu'},
        encode_kwargs={'normalize_embeddings': True}
    )
    
    # Split documents into chunks
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=512,
        chunk_overlap=50,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    
    # Process documents
    texts = []
    for doc in documents:
        chunks = text_splitter.split_text(doc)
        texts.extend(chunks)
    
    # Create vector store
    vectorstore = FAISS.from_texts(texts, embeddings)
    
    # Initialize language model
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)
    
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    # Create pipeline
    pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_length=512,
        temperature=0.7,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )
    
    llm = HuggingFacePipeline(pipeline=pipe)
    
    # Create RAG chain
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=vectorstore.as_retriever(search_kwargs={"k": 3}),
        return_source_documents=True
    )
    
    return qa_chain

# Usage example
if __name__ == "__main__":
    docs = [
        "Your document content here...",
        "More document content..."
    ]
    
    rag_chain = create_advanced_rag(docs)
    
    # Query the system
    result = rag_chain("Your question here")
    print(result['result'])
    print("\nSources:")
    for doc in result['source_documents']:
        print(doc.page_content[:200] + "...")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM, pipeline
from sentence_transformers import SentenceTransformer, util
import faiss
import numpy as np
from typing import List, Tuple

class SimpleRAGSystem:
    def __init__(self, 
                 embedding_model_name: str = 'sentence-transformers/all-MiniLM-L6-v2',
                 generation_model_name: str = 'microsoft/DialoGPT-medium'):
        """
        Initialize RAG system with embedding and generation models
        """
        # Initialize embedding model
        self.embedding_model = SentenceTransformer(embedding_model_name)
        
        # Initialize generation model
        self.tokenizer = AutoTokenizer.from_pretrained(generation_model_name)
        self.generation_model = AutoModelForCausalLM.from_pretrained(generation_model_name)
        
        # Add padding token if not present
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
            
        # Initialize vector database
        self.vector_db = None
        self.documents = []
        
    def build_vector_db(self, documents: List[str]):
        """
        Build vector database from documents
        """
        print("Building vector database...")
        self.documents = documents
        
        # Generate embeddings
        embeddings = self.embedding_model.encode(documents, convert_to_tensor=True)
        embeddings = embeddings.cpu().numpy()
        
        # Create FAISS index
        dimension = embeddings.shape[1]
        self.vector_db = faiss.IndexFlatIP(dimension)  # Inner product for cosine similarity
        
        # Normalize embeddings and add to index
        faiss.normalize_L2(embeddings)
        self.vector_db.add(embeddings.astype(np.float32))
        
        print(f"Vector database built with {len(documents)} documents")
        
    def retrieve(self, query: str, top_k: int = 3) -> List[Tuple[str, float]]:
        """
        Retrieve most relevant documents for a query
        """
        if self.vector_db is None:
            raise ValueError("Vector database not built. Call build_vector_db first.")
            
        # Generate query embedding
        query_embedding = self.embedding_model.encode([query], convert_to_tensor=True)
        query_embedding = query_embedding.cpu().numpy()
        faiss.normalize_L2(query_embedding)
        
        # Search for similar documents
        scores, indices = self.vector_db.search(query_embedding.astype(np.float32), top_k)
        
        # Return retrieved documents with scores
        retrieved_docs = []
        for i, (score, idx) in enumerate(zip(scores[0], indices[0])):
            if idx < len(self.documents):
                retrieved_docs.append((self.documents[idx], float(score)))
                
        return retrieved_docs
    
    def generate_response(self, query: str, retrieved_docs: List[Tuple[str, float]], 
                         max_length: int = 200) -> str:
        """
        Generate response using retrieved context
        """
        # Build context from retrieved documents
        context_parts = []
        for doc, score in retrieved_docs:
            if score > 0.3:  # Only include relevant documents
                context_parts.append(doc)
        
        context = " ".join(context_parts)
        
        # Build prompt
        prompt = f"""Context: {context}
        
Question: {query}

Based on the context above, please provide a comprehensive answer to the question. If the context doesn't contain enough information, mention that in your response."""
        
        # Generate response
        inputs = self.tokenizer.encode(prompt, return_tensors='pt', 
                                      max_length=512, truncation=True)
        
        with torch.no_grad():
            outputs = self.generation_model.generate(
                inputs,
                max_length=max_length,
                num_return_sequences=1,
                temperature=0.7,
                pad_token_id=self.tokenizer.pad_token_id,
                eos_token_id=self.tokenizer.eos_token_id
            )
        
        response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        # Extract only the answer part
        if "Based on the context above, please provide" in response:
            response = response.split("Based on the context above, please provide")[1].strip()
        
        return response
    
    def query(self, query: str, top_k: int = 3) -> str:
        """
        Complete RAG pipeline: retrieve and generate
        """
        # Retrieve relevant documents
        retrieved_docs = self.retrieve(query, top_k)
        
        # Generate response
        response = self.generate_response(query, retrieved_docs)
        
        return response

# Example usage
if __name__ == "__main__":
    # Sample documents (your knowledge base)
    documents = [
        "The capital of France is Paris, which is known for the Eiffel Tower and Louvre Museum.",
        "Machine learning is a subset of artificial intelligence that focuses on algorithms that can learn from data.",
        "Python is a high-level programming language created by Guido van Rossum in 1991.",
        "The Earth orbits around the Sun in approximately 365.25 days, which defines our calendar year.",
        "Artificial neural networks are computing systems inspired by biological neural networks in animal brains."
    ]
    
    # Initialize RAG system
    rag_system = SimpleRAGSystem()
    
    # Build vector database
    rag_system.build_vector_db(documents)
    
    # Query the system
    queries = [
        "What is the capital of France?",
        "Tell me about machine learning",
        "How long does Earth take to orbit the Sun?"
    ]
    
    for query in queries:
        print(f"\nQuery: {query}")
        response = rag_system.query(query)
        print(f"Response: {response}")
        print("-" * 50)